# Описание эксперимента моделирования динамики временных рядов с использованием хаотических нейронных сетей


## Схема пайплайна


```text
┌──────────────┐
│   Данные     │
│ OHLCV, тикеры│
└──────┬───────┘
       │
       v
┌──────────────────────┐
│ Предобработка данных,│  
│ лог-доходности       │
└──────┬───────────────┘
       │
       ├───────────────────────────────┐
       │                               │
       v                               v
┌──────────────────────┐      ┌──────────────────────────┐
│ Извлечение признаков │      │ Прогнозирование          │
│ (хвосты, память,     │      │ Фиксация метрик качества │
│ сложность, хаос)     │      │ прогнозирования          │
└──────┬───────────────┘      └───────────┬──────────────┘
       │                                  │
       v                                  │
┌──────────────────────────┐              │
│ Отбор и консолидация     │              │
│ финальных наборов        │              │
│ признаков                │              │
└──────┬───────────────────┘              │
       │                                  │
       |                                  │
       │                                  │
       v                                  │
┌──────────────────────┐                  │
│ Кластеризация        │                  │
│ и профилирование     │                  │
│ кластеров            │                  │
└──────┬───────────────┘                  │
       │                                  │
       │                                  │
       │                                  │
       │                                  │
       │                                  │
       v                                  v
┌──────────────────────┐        ┌────────────────────────┐
│ Аналитический этап   │───────>│ Метамоделирование      │
│                      │        │ выбор лучшей модели    │
│                      │        │ по свойствам ряда      │
└──────────────────────┘        └────────────────────────┘
```
Кластеризация в данном проекте используется как аналитический вспомогательный этап: она помогает выявлять режимы рядов и интерпретировать поведение моделей, но не является основным механизмом выбора модели. Основной итоговый механизм выбора реализован через метамоделирование.

## 2. Цели исследования
1. Анализ применимости хаотических нейронных сетей для прогнозирования финансовых временных рядов; определить, где они могут работать лучше, включая анализ посредством кластеризации.
2. Изучить, можно ли выбрать лучшую модель прогнозирования из свойств временных рядов с помощью метамоделирования (построения модели, предсказывающей лучшую модель для прогнозирования).


## 3. Структура пайплайна и репозитория

Проект организован как воспроизводимый исследовательский пайплайн, в котором отдельные этапы последовательно преобразуют исходные рыночные данные в признаки, результаты прогнозирования и итоговые артефакты метамоделирования.

Основные стадии пайплайна:

1. **Подготовка данных.**  
   Формируются каталог рядов, профили датасета и таблица лог-доходностей. На этом этапе задаётся состав исследуемой выборки и базовая структура временных рядов.

2. **Построение и отбор признаков.**  
   Для каждого ряда вычисляются группы признаков, описывающие статистические свойства, временную зависимость, структурную сложность и хаотические характеристики. Затем проводится скрининг, консолидация и сборка финальных наборов признаков.

3. **Кластеризация и профилирование кластеров.**  
   На основе финальных признаков строятся кластеризации рядов, после чего выполняется профилирование кластеров и анализ того, как различаются режимы рядов и качество моделей в этих режимах. Этот этап используется как аналитический, а не как основной механизм выбора модели.

4. **Сравнение моделей прогнозирования.**  
   На выбранной выборке запускается сравнение набора моделей прогнозирования на нескольких горизонтах. Результатом этапа становятся метрики качества по моделям, горизонтам и рядам.

5. **Метамоделирование.**  
   На финальном этапе строится модель, которая по свойствам ряда пытается выбрать наиболее подходящую модель прогнозирования. Именно этот этап является основным итоговым результатом проекта.

Структурно репозиторий разделён на несколько основных блоков:
- `src/` — реализация этапов пайплайна;
- `configs/` — конфигурации запусков;
- `artifacts/` — выходные артефакты, таблицы, отчёты;
- `docs/` — поясняющие материалы;
- `tests/` — проверка корректности ключевых этапов.

С точки зрения научного содержания центральная логика проекта выглядит так: сначала исследуются свойства рядов и поведение различных моделей прогнозирования, затем проверяется, можно ли использовать эти свойства для выбора наилучшей модели прогнозирования с помощью метамоделирования.


## 4. Данные и предварительная обработка

### 4.1. Состав исходных данных

Исходные данные проекта представляют собой набор рыночных временных рядов по российским и американским инструментам. На этапе подготовки данных формируются:
- каталог рядов;
- профили датасета;
- таблица лог-доходностей, используемая в дальнейших этапах анализа.

Краткие характеристики подготовленных данных приведены в таблице ниже.

| Показатель | Значение |
|---|---:|
| Число рядов в каталоге | 7444 |
| Число рядов US | 7195 |
| Число рядов RU | 249 |
| Размер профиля `core_balanced` | 418 |
| Размер профиля `extended_us_heavy` | 1009 |

В дальнейшем для основных экспериментов использовался профиль `core_balanced` (сбалансирован по количеству рядов RU и US), обеспечивающий более сбалансированное покрытие рядов по рынкам.

### 4.2. Группы признаков и финальный отбор

На этапе построения признаков в проекте использовались четыре содержательные группы:

1. **Признаки зависимости и памяти** (`block_a_dependence.py`)  
   Описывают долгосрочную память, автокорреляционную структуру и отклонения от случайного блуждания.

2. **Спектральные признаки и признаки сложности** (`block_b_spectrum.py`)  
   Описывают спектральную форму ряда, энтропию, алгоритмическую сложность.

3. **Признаки распределения и хвостов** (`block_c_tails.py`)  
   Описывают тяжесть хвостов, форму распределения и устойчивые характеристики экстремальных значений.

4. **Хаотические и нелинейно-динамические признаки** (`block_d_chaos.py`)  
   Предназначены для описания вложения, размерности, временной задержки и характеристик хаотической динамики.

Изначально пространство признаков было шире, однако после этапов скрининга, консолидации и формирования финальных наборов были оставлены только признаки, вошедшие в финальные таблицы `final_clustering_features_base_v1.parquet` и `final_clustering_features_with_chaos_v1.parquet`. В коде `final_feature_sets.py` зафиксирован следующий итоговый состав.

**Финальный базовый набор признаков (`22` признака):**

- **Зависимость и память**  
  `hurst_rs`, `hurst_dfa`, `acf_lag_2`, `acf_lag_5`, `acf_lag_10`, `acf_lag_25`, `acf_lag_50`, `acf_lag_100`, `abs_acf_lag_2`, `abs_acf_lag_5`, `abs_acf_lag_10`, `abs_acf_lag_25`, `abs_acf_lag_50`, `vr_q10`, `lb_ret_stat_50`

- **Спектральные признаки и признаки сложности**  
  `lz_complexity`, `permutation_entropy`, `spectral_flatness`

- **Признаки распределения и хвостов**  
  `kurtosis`, `robust_kurtosis`, `tail_ratio_upper`, `hill_tail_index`

**Дополнительные хаотические признаки (`+3` к базовому набору):**
- `correlation_dimension`
- `embedding_dimension`
- `selected_delay_tau`

Итоговые артефакты имеют вид:
- `final_clustering_features_base_v1.parquet` — `22` признака;
- `final_clustering_features_with_chaos_v1.parquet` — `25` признаков.

### 4.3. Общая схема разбиения и оценки

Для этапа прогнозирования использовалась схема с тремя временными разбиениями. Для каждого ряда, горизонта и модели формировались независимые train/test-разбиения, где обучающая часть строго предшествует тестовой.

## 5. Модели прогнозирования
- baseline: 
    `naive_zero` - прогноз 0, 
    `naive_mean` - прогноз среднего.
- classic: 
    `ridge_lag` - ридж-регрессия,
    `esn` - резервуарная модель,
    `vanilla_mlp` - многослойный перцептрон,
    `lstm_forecast` - LSTM.
- chaotic: 
    `chaotic_esn` 
    `transient_chaotic_esn`, 
    `chaotic_mlp`, 
    `chaotic_lstm_forecast`, 
    `chaotic_logistic_net`

- **`chaotic_esn`**  
  Это вариант `ESN`, в котором хаотический режим задаётся через изменение спектрального радиуса резервуара.

- **`transient_chaotic_esn`**  
  В `TransientChaoticESN` хаотичность задаётся не постоянным параметром, а переходным режимом: рекуррентный вклад масштабируется временным коэффициентом `g_t`, который убывает от `g_start` к `g_end`. То есть модель стартует в более хаотическом режиме и затем постепенно переходит к более стабильной динамике.

- **`chaotic_mlp`**  
  В `ChaoticMLP` используется хаотическая функция активация, построенная на основе logistic map

- **`chaotic_lstm_forecast`**  
  В `LSTMForecastChaotic` архитектура LSTM сохраняется, но веса инициализируются не стандартным способом, а хаотической последовательностью logistic map 

- **`chaotic_logistic_net`**  
  В `ChaoticLogisticNet` скрытая динамика непосредственно строится на логистическом отображении: состояние обновляется по формуле вида `r_t * h * (1 - h)` с демпфированием через `beta`


## 6. Постановка эксперимента по прогнозированию

Основной прогон в текущей версии проекта выполнялся на профиле `core_balanced` и включал прогнозирование на трёх горизонтах: 1, 5 и 20 шагов вперёд. Для каждого горизонта использовался свой размер входного окна

| Параметр                           | Значение                                          |
| ---------------------------------- | ------------------------------------------------- |
| Конфиг                             | `configs/forecasting_benchmark_v2.yaml`           |
| Датасет                     | `core_balanced`                                   |
| Число рядов в полном профиле       | 418                                               |
| Горизонты                          | 1, 5, 20                                          |
| Размеры окон                       | 64 / 32 / 16                                      |
| Схема разбиения                    | `rolling_origin`                                  |
| Число временных разбиений          | 3                                                 |
| Число моделей                      | 11                                                |
| Число задач                        | 41 382                                            |
| Torch MLP / chaotic MLP / logistic | `max_epochs = 30`, `patience = 5`                 |
| Torch LSTM / chaotic LSTM          | `max_epochs = 40`, `patience = 6`                 |
| Основной каталог результатов       | `artifacts/forecasting/forecasting_benchmark_v2/` |
| Каталог отчётов                    | `artifacts/reports/forecasting_audit_v2/`         |


## 7. Кластеризация и профилирование кластеров
Кластеризация использовалась как аналитический этап для изучения структурной неоднородности и моделирования поведения кластеров.
Этот этап не является окончательным механизмом выбора модели; окончательный выбор осуществляется посредством метамоделирования.
| Параметры эксперимента | Значение |
|---|---|
| `feature_set`(набор признаков) | `base` (без хаотических метрик), `with_chaos` (дополненный хаотическими метриками) |
| `scaler` | `identity`, `robust`, `quantile` |
| `space_type`(тип пространства признаков) | `original`, `pca` |
| `algorithm` | `gmm`, `agglomerative` |
| `k`(количество кластров) | 2..8 |
| `n_bootstrap` | 30 |

После выбора конфигурации выполнялось профилирование кластеров:
- анализ распределения признаков по кластерам;
- выделение ключевых различающих признаков;
- проверка баланса кластеров;
- сопоставление кластеров с качеством моделей прогнозирования.

| Конфигурация | Число кластеров | Набор признаков | Алгоритм | Пространство | Масштабирование | Silhouette | Davies–Bouldin | Calinski–Harabasz | Bootstrap stability |
|---|---:|---|---|---|---|---:|---:|---:|---:|
| Лучшая конфигурация без хаотических признаков (`cfg_0185`) | 4 | без хаотических признаков (`base`) | GMM | PCA | quantile | 0.190 | 1.839 | 71.178 | 0.850 |
| Лучшая конфигурация с хаотическими признаками (`cfg_0443`) | 3 | с хаотическими признаками (`with_chaos`) | agglomerative | PCA | quantile | 0.198 | 1.739 | 83.165 | 0.884 |

Обе конфигурации дают приемлемое и интерпретируемое разбиение рядов. Значения `silhouette` находятся на умеренном уровне, но при этом `Davies–Bouldin` и `Calinski–Harabasz` указывают на наличие различимой внутренней структуры, а bootstrap-устойчивость остаётся высокой. Это позволяет считать полученную кластеризацию аналитически полезной, даже если она не даёт идеально разделённых кластеров.

При добавлении хаотических признаков лучшая конфигурация оказывается немного сильнее по всем основным критериям качества: улучшается `silhouette`, снижается `Davies–Bouldin`, растёт `Calinski–Harabasz`, а устойчивость кластеров по bootstrap остаётся высокой. Это означает, что хаотические признаки действительно добавляют информацию о структуре рядов, по крайней мере на уровне геометрии признакового пространства.

На тепловой карте профилей кластеров для конфигурации `with_chaos` (`cfg_0443`, 3 кластера) видно, что кластеры различаются по нескольким содержательным направлениям. Один кластер характеризуется более выраженными хаотическими и хвостовыми свойствами, другой — более сильной зависимостью и памятью, третий — промежуточным профилем. То есть кластеризация действительно выделяет разные режимы динамики рядов, а не просто механически делит выборку на части.

Вместе с тем дальнейший анализ показал, что сама по себе кластеризация не дала устойчивого и достаточно сильного правила выбора модели прогнозирования. Иными словами, она оказалась полезна как инструмент исследования структуры выборки и интерпретации поведения моделей, но не как основной механизм маршрутизации между моделями прогнозирования.


Поэтому роль кластеризации в проекте можно описать так: она не дала готового правила выбора модели, но позволила:
- выделить режимы рядов;
- проверить, насколько различается качество моделей между этими режимами;
- использовать кластерную структуру как дополнительный инструмент интерпретации, в том числе при анализе применимости хаотических моделей.


## 8. Результаты прогнозирования

### 8.1. Средние значения RMSE по моделям и горизонтам

В таблице ниже приведены средние значения `RMSE` по каждому сочетанию «модель — горизонт прогнозирования». Для расчёта использовались результаты прогона `forecasting_benchmark_v2`.

| Модель | h = 1 | h = 5 | h = 20 |
|---|---:|---:|---:|
| `naive_zero` | 0.028429 | 0.059520 | 0.112945 |
| `naive_mean` | 0.028440 | 0.059651 | 0.113931 |
| `ridge_lag` | 0.028766 | 0.060872 | 0.116186 |
| `esn` | 0.028913 | 0.061184 | 0.117051 |
| `chaotic_esn` | 0.034506 | 0.070622 | 0.125896 |
| `transient_chaotic_esn` | 0.028910 | 0.061467 | 0.119251 |
| `vanilla_mlp` | 0.028867 | 0.060776 | 0.116559 |
| `chaotic_mlp` | 0.029492 | 0.060784 | 0.118274 |
| `chaotic_logistic_net` | 0.031370 | 0.062342 | 0.119614 |
| `lstm_forecast` | 0.028488 | 0.059885 | 0.114285 |
| `chaotic_lstm_forecast` | 0.028494 | 0.059981 | 0.115011 |

**Источник таблицы:** `forecasting_benchmark_forecasting_benchmark_v2.xlsx`, лист `model_summary_by_horizon`, `run_id = forecasting_benchmark_v2`.

По этой таблице видно, что:

- по среднему `RMSE` на всех горизонтах минимальное значение показывает `naive_zero`;
- на `h = 1` различия между `naive_zero`, `naive_mean`, `lstm_forecast` и `chaotic_lstm_forecast` малы, но `ESN` уже не является лучшей моделью по `RMSE`;
- на горизонтах `h = 5` и `h = 20` baseline-модели остаются наиболее сильными по среднему масштабу ошибки;
- `chaotic_esn` заметно уступает базовому `esn` по `RMSE` на всех горизонтах;
- `chaotic_lstm_forecast` близка к обычной `lstm_forecast`, но не улучшает её в среднем.

**[Место для изображения: тепловая карта средних значений RMSE по моделям и горизонтам]**

### 8.2. Wins-анализ по RMSE

Наряду со средними значениями метрик полезно рассмотреть, сколько раз каждая модель оказывалась лучшей на отдельном ряду. В таблице ниже под победой понимается случай, когда модель показала минимальное значение `RMSE` среди всех кандидатов на данном ряду и данном горизонте.

| Модель | h = 1 | h = 5 | h = 20 |
|---|---:|---:|---:|
| `naive_zero` | 182 | 185 | 180 |
| `naive_mean` | 87 | 104 | 91 |
| `ridge_lag` | 45 | 20 | 16 |
| `esn` | 15 | 7 | 3 |
| `chaotic_esn` | 0 | 0 | 5 |
| `transient_chaotic_esn` | 24 | 5 | 3 |
| `vanilla_mlp` | 1 | 12 | 28 |
| `chaotic_mlp` | 1 | 13 | 19 |
| `chaotic_logistic_net` | 0 | 2 | 14 |
| `lstm_forecast` | 29 | 33 | 32 |
| `chaotic_lstm_forecast` | 34 | 37 | 27 |

**Источник таблицы:** `forecasting_benchmark_forecasting_benchmark_v2.xlsx`, лист `model_series_summary`. Для каждого сочетания `(series_id, horizon)` выбиралась модель с минимальным `mean_rmse`.

Выводы по wins-анализу отличаются от предыдущего smoke-прогона:

- `naive_zero` является наиболее частым победителем по `RMSE` на всех горизонтах;
- `naive_mean` стабильно занимает вторую позицию по числу побед;
- `ESN` по `RMSE` больше не выглядит доминирующей моделью: число побед снижается с 15 на `h = 1` до 3 на `h = 20`;
- `lstm_forecast` и `chaotic_lstm_forecast` дают заметное число локальных побед, но не становятся лидерами;
- хаотические модели полезны как отдельные кандидаты, но по `RMSE` не формируют устойчивого преимущества.

**[Место для изображения: столбчатая диаграмма числа побед моделей по горизонтам]**

### 8.3. Лидеры по горизонтам и метрикам

| Горизонт | Лучшая модель по RMSE | RMSE | Лучшая модель по `directional_accuracy` | `directional_accuracy` |
|---:|---|---:|---|---:|
| 1 | `naive_zero` | 0.028429 | `esn` | 0.492506 |
| 5 | `naive_zero` | 0.059520 | `naive_mean` | 0.506563 |
| 20 | `naive_zero` | 0.112945 | `naive_mean` | 0.525371 |

**Источник таблицы:** `forecasting_benchmark_forecasting_benchmark_v2.xlsx`, лист `winners_by_horizon_metric`.

Эта таблица показывает расхождение между двумя типами метрик:

- по `RMSE` на всех горизонтах лидирует простая baseline-модель `naive_zero`;
- по `directional_accuracy` на коротком горизонте лучшей остаётся `esn`;
- на горизонтах 5 и 20 лучшей по направлению становится `naive_mean`;
- следовательно, в новом прогоне нельзя утверждать, что `ESN` является универсальным лидером даже по направленческой метрике.

### 8.4. Сравнение групп моделей

| Горизонт | Группа моделей | Средний RMSE | Средний `directional_accuracy` |
|---:|---|---:|---:|
| 1 | `baseline` | 0.028434 | 0.263055 |
| 1 | `classic` | 0.028758 | 0.487956 |
| 1 | `chaotic` | 0.030554 | 0.484870 |
| 5 | `baseline` | 0.059585 | 0.262399 |
| 5 | `classic` | 0.060679 | 0.499488 |
| 5 | `chaotic` | 0.063039 | 0.498207 |
| 20 | `baseline` | 0.113438 | 0.266335 |
| 20 | `classic` | 0.116020 | 0.515098 |
| 20 | `chaotic` | 0.119609 | 0.510296 |

**Источник таблицы:** агрегирование `forecasting_benchmark_forecasting_benchmark_v2.xlsx`, лист `model_summary_by_horizon`.

В текущем запуске:

- по среднему `RMSE` сильнее всего выглядит группа `baseline`;
- по среднему `directional_accuracy` baseline-модели слабы из-за `naive_zero`, который почти не предсказывает направление;
- группы `classic` и `chaotic` близки по `directional_accuracy`, но `classic` немного лучше в среднем на всех горизонтах;
- хаотические модели не дают среднего преимущества ни по `RMSE`, ни по `directional_accuracy`, но остаются сопоставимыми с классическими моделями по направлению.

### 8.5. Распределение побед по группам

| Горизонт | Метрика | Доля побед `baseline` | Доля побед `classic` | Доля побед `chaotic` |
|---:|---|---:|---:|---:|
| 1 | `RMSE` | 0.6435 | 0.2153 | 0.1411 |
| 1 | `directional_accuracy` | 0.1555 | 0.3947 | 0.4498 |
| 5 | `RMSE` | 0.6914 | 0.1722 | 0.1364 |
| 5 | `directional_accuracy` | 0.2536 | 0.2823 | 0.4641 |
| 20 | `RMSE` | 0.6483 | 0.1890 | 0.1627 |
| 20 | `directional_accuracy` | 0.2632 | 0.2895 | 0.4474 |

**Источник таблицы:** расчёт по `forecasting_benchmark_forecasting_benchmark_v2.xlsx`, лист `model_series_summary`; победитель выбирался отдельно для каждой пары `(series_id, horizon)`.

Эта таблица важна, потому что она показывает не среднее качество группы, а частоту локальных побед:

- по `RMSE` baseline-группа доминирует на всех горизонтах;
- по `directional_accuracy` чаще всего побеждают хаотические модели;
- при этом средняя `directional_accuracy` у группы `classic` немного выше, чем у `chaotic`, то есть хаотические модели чаще выигрывают на отдельных рядах, но не дают лучшего среднего результата;
- это усиливает аргумент в пользу метамоделирования: разные типы моделей выигрывают на разных рядах и по разным метрикам.

## 10. Задача метамоделирования

На заключительном этапе проекта рассматривается задача выбора наилучшей модели прогнозирования по свойствам временного ряда. Для каждой пары `(горизонт, метрика)` формируется отдельная задача: по признакам ряда нужно предсказать, какая из моделей-кандидатов даст наилучший результат.

В проекте использовались две целевые метрики:

- `RMSE` — для оценки масштаба ошибки прогноза;
- `directional_accuracy` — для оценки правильности предсказания направления движения.

Для интерпретации результатов метамоделирования используются три опорные величины:

1. **`best single`** — одна фиксированная модель прогнозирования, которая в среднем показывает лучший результат на данной задаче и затем применяется ко всем рядам без индивидуального выбора.

2. **`oracle`** — недостижимый ориентир, при котором для каждого ряда выбирается фактически лучшая модель уже по известному результату. Эта величина задаёт верхнюю границу качества.

3. **`achieved metric`** — фактическое качество, достигнутое метамоделью, которая выбирает модель прогнозирования по признакам ряда.

Сравнение этих величин позволяет ответить на два вопроса:

- даёт ли метамоделирование практический выигрыш по сравнению с использованием одной глобально лучшей модели;
- насколько далеко текущий результат находится от теоретического предела, задаваемого `oracle`.

При интерпретации улучшения важно учитывать знак метрики:

- для `RMSE` улучшение считается как `best_single - achieved`, то есть положительное значение означает уменьшение ошибки;
- для `directional_accuracy` улучшение считается как `achieved - best_single`, то есть положительное значение означает рост точности.

## 11. Конфигурации метамоделирования

| parameter | values |
|---|---|
| `candidate_set` | `top_3`, `top_4`, `top_5`, `top_6` |
| `feature_set` | `full` |
| `balancing_mode` | `default`, `balanced` |
| `decision_rule` | `top_1`, `confidence_fallback_0.50`, `confidence_fallback_0.60`, `confidence_fallback_0.70`, `confidence_fallback_0.80` |
| classifiers | `logistic_regression`, `random_forest_classifier`, `catboost_classifier` |
| repeats | 5 |

**Источник таблицы:** `meta_modeling_experiments_v2.xlsx`, лист `summary`.

## 12. Результаты метамоделирования

| horizon | metric | best configuration | achieved | best_single | oracle | improvement | gap_to_oracle |
|---:|---|---|---:|---:|---:|---:|---:|
| 1 | `rmse` | `catboost_classifier`, `top_4`, `full`, `default`, `confidence_fallback_0.50` | 0.027126 | 0.027127 | 0.027058 | 0.000002 | 0.000068 |
| 1 | `directional_accuracy` | `logistic_regression`, `top_3`, `full`, `default`, `top_1` | 0.495035 | 0.492753 | 0.506111 | 0.002282 | 0.011076 |
| 5 | `rmse` | `catboost_classifier`, `top_4`, `full`, `balanced`, `confidence_fallback_0.50` | 0.057901 | 0.057903 | 0.057689 | 0.000002 | 0.000212 |
| 5 | `directional_accuracy` | `logistic_regression`, `top_3`, `full`, `balanced`, `top_1` | 0.510367 | 0.506978 | 0.527955 | 0.003389 | 0.017588 |
| 20 | `rmse` | `random_forest_classifier`, `top_6`, `full`, `default`, `top_1` | 0.111225 | 0.111258 | 0.109370 | 0.000034 | 0.001855 |
| 20 | `directional_accuracy` | `logistic_regression`, `top_3`, `full`, `default`, `top_1` | 0.526316 | 0.524608 | 0.547205 | 0.001709 | 0.020889 |

**Источник таблицы:** `meta_modeling_experiments_v2.xlsx`, лист `summary`. Для каждой пары `(horizon, target_metric)` выбрана конфигурация с максимальным `improvement_mean`.

Выводы:

- по `RMSE` улучшения есть, но они крайне малы;
- наиболее заметный выигрыш по `RMSE` возникает на `h = 20`, но даже там improvement равен только `0.000034`;
- по `directional_accuracy` метамоделирование даёт более содержательный эффект;
- лучший прирост по направлению наблюдается на `h = 5`: `+0.003389`;
- во всех задачах сохраняется разрыв до `oracle`, особенно по `directional_accuracy`.

### 12.1. Сводка по числу конфигураций и положительным улучшениям

| horizon | metric | total_configs | positive_improvement | zero_improvement | mean_majority_class_share |
|---:|---|---:|---:|---:|---:|
| 1 | `rmse` | 120 | 16 | 51 | 0.321 |
| 1 | `directional_accuracy` | 120 | 43 | 41 | 0.314 |
| 5 | `rmse` | 120 | 19 | 46 | 0.355 |
| 5 | `directional_accuracy` | 120 | 43 | 52 | 0.323 |
| 20 | `rmse` | 120 | 19 | 60 | 0.441 |
| 20 | `directional_accuracy` | 120 | 44 | 46 | 0.344 |

**Источник таблицы:** `meta_modeling_experiments_v2.xlsx`, листы `summary` и `candidates`.

`mean_majority_class_share` показывает среднюю долю наиболее частого класса-победителя в задаче выбора модели. Чем выше значение, тем сильнее вырождение задачи: одна модель чаще оказывается лучшей, и метамодели сложнее получить дополнительный выигрыш.

В новом эксперименте:

- сильного классового вырождения на `h = 1` уже нет: majority share около `0.32`, а не около `0.99`, как в старом smoke-прогоне;
- наиболее выраженное доминирование класса наблюдается для `RMSE` на `h = 20`, где `mean_majority_class_share = 0.441`;
- количество конфигураций с положительным улучшением умеренное: от 16 до 44 из 120;
- по `directional_accuracy` положительных конфигураций больше, чем по `RMSE`.

## 13. Интерпретация результатов

- Универсальной модели для всех горизонтов и метрик не наблюдается.
- По среднему `RMSE` в новом forecasting-прогоне сильнее всего выглядят baseline-модели, прежде всего `naive_zero`.
- `ESN` больше нельзя описывать как ключевую модель проекта в целом: он лидирует только по `directional_accuracy` на `h = 1`.
- На горизонтах `h = 5` и `h = 20` по средней `directional_accuracy` лучшей становится `naive_mean`.
- Хаотические модели не улучшают средние значения `RMSE`, но часто выигрывают по `directional_accuracy` на отдельных рядах.
- Метамоделирование даёт слабый эффект по `RMSE`, потому что выигрыш над `best single` минимален.
- По `directional_accuracy` эффект метамоделирования заметнее, особенно на горизонте `h = 5`.
- Разрыв до `oracle` остаётся существенным, значит признаки ряда пока не позволяют стабильно восстановить фактически лучшую модель.
- Практический смысл метамоделирования в текущем виде — не в радикальном снижении ошибки, а в маршрутизации между моделями там, где локальные победители различаются по рядам.